# Playground

Look at a built dataset, the prompts the simulator gets, and results of a run. Everything comes from the `nsg_sim` package; datasets are built with `nsg build` and runs with `nsg run` (see `docs/ADDING_A_DATASET.md`).

In [ ]:
import json, random
from collections import Counter

from nsg_sim.storage import list_datasets, load_dataset
from nsg_sim.prompts.build import build_prompt

for d in list_datasets():
    print(f"{d['name']:<28} {d['n_records']:>6}  {d['description']}")

## A dataset

In [ ]:
DATASET = 'triage_simple_v1'
records, manifest = load_dataset(DATASET)
print(json.dumps(manifest['stats'], indent=1)[:800])

In [ ]:
# explanation texts of one rule at every level
rule_id = records[0]['rule_id']
seen = {}
for r in records:
    if r['rule_id'] == rule_id:
        seen.setdefault((r['explanation_type'], r['level']), r['explanation_text'])
print(records[0]['rule_formal'])
for (etype, level), text in seen.items():
    print(f'{etype:<14} {level}  {text}')

In [ ]:
# one record and the prompt built from it
r = random.Random(0).choice(records)
print({k: r[k] for k in ('record_id', 'label_x', 'label_cx', 'explanation_label_cx', 'discriminating')})
print()
print(build_prompt(r, manifest['domain']['task'], with_explanation=True))

## Results of a run

`nsg run configs/experiments/triage_simple_smoke.yaml` gives a free example with baseline strategies.

In [ ]:
from pathlib import Path
from nsg_sim.analysis.metrics import load_pairs, group, accuracy, nsg, cluster_ci

RUN = Path('../runs/triage_simple_smoke')
pairs = load_pairs([RUN])
for key, cell in sorted(group([p for p in pairs if p['evidence_condition'] == 'none'], ['simulator', 'explanation_type']).items()):
    lo, hi = cluster_ci(cell, nsg)
    print(f"{key[0]:<18} {key[1]:<16} acc {accuracy(cell):.2f}  NSG {nsg(cell):+.2f} [{lo:+.2f}, {hi:+.2f}]  n={len(cell)}")

In [ ]:
from nsg_sim.analysis.plots import plot_metric

fig = plot_metric([p for p in pairs if p['evidence_condition'] == 'none'], metric='acc', x='level',
                  lines='explanation_type', panels='simulator')
fig